# Speech Emotion Recognition — wav2vec2 + RAVDESS

Fine-tune `facebook/wav2vec2-base-960h` for **8-class** emotion classification on the RAVDESS dataset using the official Hugging Face `Trainer` API.

---

## How to run

| Environment | What to run |
|-------------|-------------|
| **This server (recommended)** | From the project root: `python run_training.py` |
| **Colab / interactive** | Run the cells below step by step |

`run_training.py` handles everything automatically:
1. Scans `../RAVDESS_DATA/ravdess/` and writes `metadata_local.csv` (fixes Windows paths + label casing from the old Colab CSV)
2. Speaker-independent 80/10/10 split by actor
3. Fine-tunes wav2vec2 with a frozen CNN encoder
4. Evaluates on the test set and saves `outputs/confusion_matrix.png`

---

## Pipeline overview

`prepare metadata` → `speaker-independent split` → `HF Dataset + feature extraction` → `Trainer fine-tuning` → `test evaluation + confusion matrix` → `save model to outputs/wav2vec2-ser-ravdess/`

After training, launch the demo: `python app/gradio_app.py`

## 1. Install dependencies

**Local server:** dependencies are installed in the project venv (see README). Skip this cell.

**Colab:** uncomment and run the pip cell below.

In [ ]:
%pip install -q -r ../requirements.txt

## 1b. (Local only) Prepare metadata

On this server, `run_training.py` does this automatically. If you are stepping through the notebook manually, run:

```python
from scripts.prepare_data import prepare
prepare()  # writes ../RAVDESS_DATA/metadata_local.csv
```

## 2. Imports and project setup

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from transformers import TrainingArguments, Trainer

from src import config
from src.dataset import (
    load_metadata,
    split_metadata,
    build_hf_datasets,
    get_feature_extractor,
    preprocess_batch,
)
from src.model import build_model
from src.utils import set_seed, compute_metrics
from src.train import DataCollatorSER, build_training_args
from src.evaluate import run_evaluation

set_seed(config.SEED)
print("Labels:", config.LABELS)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## 3. Load metadata and create a speaker-independent split
We split by **actor**, not by row, so no speaker appears in both train and test — this gives a realistic estimate of generalization.

In [ ]:
df = load_metadata(config.METADATA_CSV)
print(f"Total clips: {len(df)}")
display(df.head())

train_df, val_df, test_df = split_metadata(df)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print("Train label distribution:")
print(train_df["emotion"].value_counts())

## 4. Build Hugging Face datasets (audio decoding)

In [ ]:
train_ds, val_ds, test_ds = build_hf_datasets(train_df, val_df, test_df)
train_ds

## 5. Load the wav2vec2 feature extractor and preprocess audio
Each clip is resampled to 16kHz and padded/truncated to a fixed length so batches can be stacked into tensors.

In [ ]:
feature_extractor = get_feature_extractor(config.MODEL_CHECKPOINT)

map_kwargs = dict(batched=True, batch_size=8, remove_columns=["audio", "emotion"])
train_ds = train_ds.map(lambda b: preprocess_batch(b, feature_extractor), **map_kwargs)
val_ds = val_ds.map(lambda b: preprocess_batch(b, feature_extractor), **map_kwargs)
test_ds = test_ds.map(lambda b: preprocess_batch(b, feature_extractor), **map_kwargs)

train_ds

## 6. Build the model
`Wav2Vec2ForSequenceClassification` adds a mean-pooling + linear classification head on top of wav2vec2's hidden states. The convolutional feature encoder is frozen by default (recommended for small datasets like RAVDESS).

In [ ]:
model = build_model(config.MODEL_CHECKPOINT)
sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())

## 7. Configure the Trainer

In [ ]:
data_collator = DataCollatorSER(feature_extractor=feature_extractor)
training_args = build_training_args()

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
try:
    trainer = Trainer(processing_class=feature_extractor, **trainer_kwargs)  # transformers >= 4.46
except TypeError:
    trainer = Trainer(tokenizer=feature_extractor, **trainer_kwargs)  # transformers < 4.46

## 8. Train

In [ ]:
trainer.train()

## 9. Validation metrics

In [ ]:
val_metrics = trainer.evaluate(val_ds)
val_metrics

## 10. Test-set evaluation, classification report and confusion matrix

In [ ]:
results = run_evaluation(trainer, test_ds, output_dir=config.OUTPUT_DIR)

In [ ]:
from IPython.display import Image, display
display(Image(filename=f"{config.OUTPUT_DIR}/confusion_matrix.png"))

## 11. Save the final model and feature extractor
This produces a directory that can be loaded directly by `app/gradio_app.py` or shared on the Hugging Face Hub.

In [ ]:
trainer.save_model(config.MODEL_DIR)
feature_extractor.save_pretrained(config.MODEL_DIR)
print(f"Model saved to {config.MODEL_DIR}")

## 12. Quick sanity-check inference on a single test clip

In [ ]:
import numpy as np

sample = test_ds[0]
input_values = torch.tensor(sample["input_values"]).unsqueeze(0).to(model.device)

with torch.no_grad():
    logits = model(input_values=input_values).logits
    pred_id = int(torch.argmax(logits, dim=-1))

print("Predicted:", config.ID2LABEL[pred_id])
print("True label:", config.ID2LABEL[sample["label"]])